In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
import numpy as np
from src.params import *

from src.preprocess import *

import torch
from torch import nn
from torch.utils.data import Dataset, Subset
from sklearn.model_selection import StratifiedGroupKFold

In [4]:
df = pd.read_csv(".." /DATA_DIR / "train.csv")

In [5]:
df

,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106795,351917269,6,12.0,2147388374,6,12.0,4195677307,10351,LRDA,0,0,0,3,0,0
106796,351917269,7,14.0,2147388374,7,14.0,290896675,10351,LRDA,0,0,0,3,0,0
106797,351917269,8,16.0,2147388374,8,16.0,461435451,10351,LRDA,0,0,0,3,0,0
106798,351917269,9,18.0,2147388374,9,18.0,3786213131,10351,LRDA,0,0,0,3,0,0


In [ ]:
mean_test = 0.237168
std_test =  0.222340

# Construction custom dataset

In [ ]:
class BrainDataset(Dataset):

    def __init__(self, metadata, mean, std):

        super().__init__()
        self.metadata = metadata #metadata is the train.csv file
        self.mean = mean #mean and std stream-calculated in the datamodule over the train set
        self.std = std

    def len(self):
        return len(self.metadata)

    def __getitem__(self, idx):

        #get file name
        spec_id = self.metadata.iloc[idx]["spectrogram_id"]
        subsample_id = self.metadata.iloc[idx]["spectrogram_sub_id"]

        #load file
        path = PROCESSED_DIR / f"{spec_id}-{subsample_id}.npy"
        spec = np.load(path)

        #log transfo
        spec = np.log1p(spec) #should be before norm?

        #normalization
        spec = (spec - self.mean)/self.std # add epsilon to avoid break if self.std = 0?

        #conversion en tensor
        spec = torch.tensor(spec, dtype= torch.float32)

        #get votes
        votes = self.metadata.iloc[idx][VOTE_COL].values
        #convert into dist
        votes = votes/votes.sum()
        #convert into tensor
        votes = torch.tensor(votes, dtype= torch.float32)


        return spec, votes

In [ ]:
test_dataset = BrainDataset(metadata= df, mean= )